# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print main metadata fields
print('Dataset Name:', dataset.metadata.name)
print('---')
print('Description:', dataset.metadata.description)
print('\nCite as:', dataset.metadata.citeAs)

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers. This helps understand which tables are available and what variables/columns each contains.

In [ ]:
# List all record sets by their @id and name for exploration
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
all_recordsets_info = []
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print(f"  description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    field_ids = [field.id for field in rs.fields] if hasattr(rs, 'fields') else []
    print(f"  Fields (@id): {field_ids}")
    print()
    all_recordsets_info.append({'@id': rs.id, 'name': getattr(rs, 'name', None), 'fields': field_ids})

# For demonstration, print fields of the first record set in detail (if exists)
if record_sets:
    example_rs = record_sets[0]
    print(f"\nExample Record Set: {example_rs.id}")
    for field in example_rs.fields:
        print(f"  Field @id: {field.id}")
        print(f"    name: {field.name if hasattr(field, 'name') else 'N/A'}")
        print(f"    dataType: {getattr(field, 'dataType', 'N/A')}")
        print()

## 3. Data Extraction
You can extract data from each record set using its `@id`. This section loads data from all available record sets into pandas DataFrames keyed by their `@id`.

You can inspect the columns (fields) and preview the first few records.

In [ ]:
# Extract data for all record sets
dataframes = {}
for rs in record_sets:
    key = rs.id
    records = list(dataset.records(record_set=key))
    if records:
        df = pd.DataFrame(records)
        dataframes[key] = df
        print(f"Record set {key} loaded with shape {df.shape}")
    else:
        print(f"Record set {key} is empty or has non-tabular structure.")

# If at least one DataFrame is loaded, display its columns and preview
if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"\nExample DataFrame for record set: {example_rs_id}")
    print("Columns (@id):", dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())  # display() for Jupyter friendly output
else:
    print("No tabular record sets were loaded. Check Croissant schema or dataset availability.")

## 4. Exploratory Data Analysis (EDA)
Apply some common data processing steps:
- Select a numeric field by its `@id`.
- Filter the DataFrame based on its value.
- Normalize the field (z-score).
- Optionally group by another field to show stratified summaries.

> **Note:** Replace the numeric field and group field with valid `@id` values as discovered in the previous steps.

In [ ]:
# Choose record set and field @ids (update to match actual IDs printed above)
record_set_id = None
numeric_field_id = None
group_field_id = None

# Automatically select a record set with numeric fields, if available
for rs in record_sets:
    key = rs.id
    if key in dataframes and not dataframes[key].empty:
        example_df = dataframes[key]
        # Pick a field with numeric dtype
        for col in example_df.columns:
            if pd.api.types.is_numeric_dtype(example_df[col]):
                record_set_id = key
                numeric_field_id = col
                break
        if record_set_id:
            # Now, for grouping, pick a non-numeric field
            for col in example_df.columns:
                if col != numeric_field_id and not pd.api.types.is_numeric_dtype(example_df[col]):
                    group_field_id = col
                    break
        if record_set_id and numeric_field_id:
            break

if not (record_set_id and numeric_field_id):
    print("No numeric field for EDA found. Adjust field selection as needed.")
else:
    df = dataframes[record_set_id]
    # Set threshold as median for demonstration
    threshold = df[numeric_field_id].median() if not df[numeric_field_id].empty else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize column
    field_norm = numeric_field_id + '_normalized'
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, field_norm]].head())
    # Group by group_field_id (if exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between key fields. For tabular data, you can plot histograms and groupwise summaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}\nin record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # Boxplot grouped by group field (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric or group field available for plotting!')

## 6. Conclusion
In this notebook, you:
- Explored metadata and available record sets and fields using mlcroissant, referencing all entities by their `@id`.
- Loaded the tabular data using record set `@id`s.
- Ran exploratory analysis and produced basic visualizations using field `@id`s.

You can now further analyze, join, or visualize clinical and molecular characteristics in this FAIR-compliant dataset for your own research questions.